In [9]:
# load data
import numpy as np
lfp_path = r"F:\GitHub\rippl-AI\allen_data\lfp_use_754312389_756781561.npy"
time_path = r"F:\GitHub\rippl-AI\allen_data\lfp_use_time_754312389_756781561.npy"
speed_path = r"F:\GitHub\rippl-AI\allen_data\lfp_use_speed_754312389_756781561.npy"
LFPs = np.load(lfp_path)  # Load LFP data from .npy file
time = np.load(time_path)  # Load time data from .npy file
speed = np.load(speed_path)

# time = np.arange(0, 10, 0.001)  # 10 seconds at 1000 Hz
# LFPs = np.random.randn(len(time), 4)  # 4 channels of LFP data
# speed = np.abs(np.random.randn(len(time)))  # Animal speed
sampling_frequency = 1250  # Hz

In [10]:
from ripple_detection import Kay_ripple_detector

# Detect ripples
ripple_times = Kay_ripple_detector(
    time, LFPs, speed, sampling_frequency,
    speed_threshold=4.0,        # cm/s
    minimum_duration=0.015,     # seconds
    zscore_threshold=2.0
)

print(ripple_times)

               start_time     end_time  duration  max_thresh  mean_zscore  \
event_number                                                                
1               36.804060    36.868860    0.0648    3.069317     1.527994   
2               36.969660    37.102460    0.1328    2.818741     1.447644   
3               50.394460    50.482460    0.0880    3.401983     1.689291   
4               54.496860    54.602460    0.1056    3.777947     1.828583   
5               61.255260    61.416060    0.1608    2.510044     1.088624   
...                   ...          ...       ...         ...          ...   
3710          9189.368149  9189.556949    0.1888    5.407291     2.283485   
3711          9189.632149  9189.812149    0.1800    2.267775     1.146147   
3712          9191.095349  9191.198549    0.1032    2.488164     1.477298   
3713          9193.216949  9193.316149    0.0992    5.814805     2.613207   
3714          9193.947349  9194.421749    0.4744    8.709958     2.996572   

In [12]:
from ripple_detection import Karlsson_ripple_detector

# Detect ripples with custom parameters
ripples = Karlsson_ripple_detector(
    time, LFPs[:,4:5], speed, sampling_frequency,
    speed_threshold=4.0,
    minimum_duration=0.015,
    zscore_threshold=3.0,
    smoothing_sigma=0.004,
    close_ripple_threshold=0.0
)

# Access detailed statistics
print(f"Detected {len(ripples)} ripple events")
print(f"Mean duration: {ripples['duration'].mean():.3f} seconds")
print(f"Mean z-score: {ripples['mean_zscore'].mean():.2f}")

Detected 1748 ripple events
Mean duration: 0.198 seconds
Mean z-score: 2.89


In [ ]:
# Load data from the ripple_AI dataset
import math
import numpy as np
def loadChunk(fid, nChannels, channels, nSamples, precision):
    size = int(nChannels * nSamples * precision)
    nSamples = int(nSamples)

    data = fid.read(size)

    # fromstring to read the data as int16
    # reshape to give it the appropiate shape (nSamples x nChannels)
    data = np.fromstring(data, dtype=np.int16).reshape(nSamples, len(channels))
    data = data[:, channels]

    return data
def bz_LoadBinary(filename, nChannels, channels, sampleSize, verbose=False):

    if (len(channels) > nChannels):
        print("Cannot load specified channels (listed channel IDs inconsistent with total number of channels).")
        return

    #aqui iria CdE de filename
    with open(filename, "rb") as f:
        dataOffset = 0

        # Determine total number of samples in file
        fileStart = f.tell()
        if verbose:
            print("fileStart ", fileStart)
        status = f.seek(0, 2) # Go to the end of the file
        fileStop = f.tell()
        f.seek(0, 0) # Back to the begining
        if verbose:
            print("fileStop ", fileStop)

        # (floor in case all channels do not have the same number of samples)
        maxNSamplesPerChannel = math.floor(((fileStop-fileStart)/nChannels/sampleSize))
        nSamplesPerChannel = maxNSamplesPerChannel

        # For large amounts of data, read chunk by chunk
        maxSamplesPerChunk = 10000
        nSamples = int(nSamplesPerChannel*nChannels)

        if verbose:
            print("nSamples ", nSamples)

        if nSamples <= maxNSamplesPerChannel:
            data = loadChunk(f, nChannels, channels, nSamples, sampleSize)
        else:
            # Determine chunk duration and number of chunks
            nSamplesPerChunk = math.floor(maxSamplesPerChunk/nChannels)*nChannels
            nChunks = math.floor(nSamples/nSamplesPerChunk)

            if verbose:
                print("nSamplesPerChannel ", nSamplesPerChannel)
                print("nSamplesPerChunk ", nSamplesPerChunk)

            # Preallocate memory
            data = np.zeros((nSamplesPerChannel,len(channels)), dtype=np.int16)

            if verbose:
                print("size data ", np.size(data, 0))

            # Read all chuncks
            i = 0
            for j in range(nChunks):
                d = loadChunk(f, nChannels, channels, nSamplesPerChunk/nChannels, sampleSize)
                m = np.size(d, 0)

                if m == 0:
                    break

                data[i:i+m, :] = d
                i = i+m

            # If the data size is not a multiple of the chunk size, read the remainder
            remainder = nSamples - nChunks*nSamplesPerChunk
            if remainder != 0:
                d = loadChunk(f, nChannels, channels, remainder/nChannels, sampleSize)
                m = np.size(d, 0)

                if m != 0:
                    data[i:i+m, :] = d

    return data